# Teacher Profiles & IAA

Note on terminology: sometimes this notebook says "unit" to refer to a shared/IOU-clustered key moment.

In [ ]:
import sys
import json
from collections import defaultdict, Counter
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import krippendorff
from adjustText import adjust_text
from IPython.display import HTML, display
from scipy.stats import wasserstein_distance

REPO_ROOT = Path("..").resolve()
sys.path.insert(0, str(REPO_ROOT))

from annotator.core.utils import compute_iou

GT_DIR = REPO_ROOT / "data" / "ground_truth_v2"

EFFECTIVENESS_LABELS = ["effective", "partial", "ineffective"]
VALID_LABELS         = set(EFFECTIVENESS_LABELS)
IOU_THRESHOLD        = 0.3
MIN_MOMENTS          = 50

COLORS = {
    "effective":   "#4caf50",
    "partial":     "#ff9800",
    "ineffective": "#f44336",
}
LABEL_ORDER = ["effective", "partial", "ineffective"]


def _cluster_moments(moments):
    """Group moments from different annotators into IoU-based connected-component clusters."""
    if not moments:
        return []
    n      = len(moments)
    parent = list(range(n))

    def _find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for i in range(n):
        for j in range(i + 1, n):
            if moments[i]["annotator_id"] == moments[j]["annotator_id"]:
                continue
            if compute_iou(
                (moments[i]["turn_start"], moments[i]["turn_end"]),
                (moments[j]["turn_start"], moments[j]["turn_end"]),
            ) >= IOU_THRESHOLD:
                ri, rj = _find(i), _find(j)
                if ri != rj:
                    parent[ri] = rj

    clusters = defaultdict(list)
    for i in range(n):
        clusters[_find(i)].append(moments[i])
    return list(clusters.values())


def build_reliability_matrix(ann_type):
    """
    Build a Krippendorff reliability matrix for the given annotation type.
    Returns (matrix, annotators) where matrix is shape (n_raters, n_units).
    Each column is one IoU-clustered moment group; values are ordinal label
    codes 0/1/2 (effective/partial/ineffective), NaN where not rated.
    """
    cat_idx = {c: i for i, c in enumerate(EFFECTIVENESS_LABELS)}
    units   = []

    for f in GT_DIR.glob("*.json"):
        with open(f) as fp:
            d = json.load(fp)
        moments = [
            m for m in d.get("key_moments", [])
            if m.get("annotation_type") == ann_type
            and m.get("strategy_label") in VALID_LABELS
        ]
        for cluster in _cluster_moments(moments):
            if len({m["annotator_id"] for m in cluster}) < 2:
                continue  # single-annotator cluster — no IAA signal
            unit = {}
            for m in cluster:
                if m["annotator_id"] not in unit:  # first label wins per annotator
                    unit[m["annotator_id"]] = cat_idx[m["strategy_label"]]
            units.append(unit)

    if not units:
        return np.empty((0, 0)), []

    all_annotators = sorted({a for u in units for a in u})
    ann_idx        = {a: i for i, a in enumerate(all_annotators)}
    matrix         = np.full((len(all_annotators), len(units)), np.nan)
    for j, unit in enumerate(units):
        for ann_id, code in unit.items():
            matrix[ann_idx[ann_id], j] = code

    return matrix, all_annotators


def submatrix_for(full_matrix, all_annotators, subset):
    """
    Extract rows for `subset` annotators and drop columns where fewer than
    2 of those annotators have a rating (no IAA signal in those columns).
    """
    ann_idx = {a: i for i, a in enumerate(all_annotators)}
    rows    = [ann_idx[a] for a in subset if a in ann_idx]
    if not rows:
        return None
    mat  = full_matrix[rows, :]
    mask = np.sum(~np.isnan(mat), axis=0) >= 2
    return mat[:, mask] if mask.any() else None


def compute_iaa(matrix):
    """Krippendorff's alpha (ordinal) from a raters × units reliability matrix."""
    if matrix is None or matrix.size == 0:
        return None
    try:
        return round(
            krippendorff.alpha(reliability_data=matrix, level_of_measurement="ordinal"),
            4,
        )
    except ValueError:
        return 1.0  # single-value domain = perfect agreement


_iaa_cache = {}

def get_reliability_matrix(ann_type):
    """Build (and cache) the reliability matrix for ann_type."""
    if ann_type not in _iaa_cache:
        _iaa_cache[ann_type] = build_reliability_matrix(ann_type)
    return _iaa_cache[ann_type]


In [ ]:
# Load ground truth and aggregate per-annotator, per-type label counts
counts = defaultdict(lambda: defaultdict(Counter))  # [ann_id][ann_type][label]

for f in GT_DIR.glob("*.json"):
    with open(f) as fp:
        d = json.load(fp)
    for m in d.get("key_moments", []):
        label = m["strategy_label"]
        if label not in VALID_LABELS:
            continue
        counts[m["annotator_id"]][m["annotation_type"]][label] += 1

print(f"Annotators: {len(counts)}")
for ann_type in ("scaffolding", "rapport"):
    n = sum(1 for c in counts.values() if ann_type in c)
    print(f"  With {ann_type} annotations: {n}")

In [ ]:
# Total moments per type (from the already-loaded counts)
total_moments = {
    ann_type: sum(sum(tc[ann_type].values()) for tc in counts.values() if ann_type in tc)
    for ann_type in ("scaffolding", "rapport")
}

for ann_type in ("scaffolding", "rapport"):
    total    = total_moments[ann_type]
    full_mat, _ = get_reliability_matrix(ann_type)
    n_units  = full_mat.shape[1] if full_mat.size > 0 else 0

    if n_units > 0:
        ann_per_unit = (~np.isnan(full_mat)).sum(axis=0).astype(int)
        n_in_multi   = int(ann_per_unit.sum())
    else:
        ann_per_unit = np.array([], dtype=int)
        n_in_multi   = 0

    # One entry per individual moment. Moments in a shared unit get the cluster
    # size as their value; moments not in any shared unit get 1.
    n_single  = total - n_in_multi
    full_dist = np.concatenate([
        np.repeat(ann_per_unit, ann_per_unit),
        np.ones(n_single, dtype=int),
    ])

    print(f"{ann_type}: {total:,} key moments total")
    print(f"  Shared moments (2+ annotators): {n_units:,}  "
          f"({100 * n_in_multi / total:.1f}% of moments are in a shared moment)")
    print(f"  Annotators per moment — "
          f"mean: {full_dist.mean():.1f}, "
          f"median: {int(np.median(full_dist))}, "
          f"max: {full_dist.max()}")

    fig, ax = plt.subplots(figsize=(6, 3))
    unique, cnts = np.unique(full_dist, return_counts=True)
    ax.bar(unique, cnts, color="#546e7a", edgecolor="white", linewidth=0.5)
    ax.set_xlabel("Annotators per moment")
    ax.set_ylabel("Number of moments")
    ax.set_title(f"{ann_type.capitalize()}: annotators per moment (including single-annotator)")
    ax.set_xticks(unique)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.show()

print("\n---- Overall IAA ----")
for ann_type in ("scaffolding", "rapport"):
    full_mat, _ = get_reliability_matrix(ann_type)
    alpha   = compute_iaa(full_mat)
    n_units = full_mat.shape[1] if full_mat.size > 0 else 0
    print(f"{ann_type}: overall Krippendorff's α = {alpha} ({n_units} units)")


For **rapport**, moments typically get multiple (2+) annotations each. 

We can calculate the average effectiveness score per moment, averaged over all annotators who annotated that moment. Mapping: effective=1, partial=0, ineffective=-1. 

This gives a continuous sense of what moments tend be less effective, and which ones tend to be more effective. 

In [ ]:
SCORE_MAP = {0: 1, 1: 0, 2: -1}  # ordinal code → score

for ann_type in ("scaffolding", "rapport"):
    full_mat, _ = get_reliability_matrix(ann_type)

    # Remap codes to scores (NaN stays NaN)
    # Fill NaN with 0 only for the cast, then restore NaN
    _safe = np.where(np.isnan(full_mat), 0, full_mat).astype(int)
    score_mat = np.where(np.isnan(full_mat), np.nan,
                        np.vectorize(SCORE_MAP.get)(_safe).astype(float))

    unit_means = np.nanmean(score_mat, axis=0)  # one value per unit

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(unit_means, bins=30, color="#546e7a", edgecolor="white", linewidth=0.5)
    ax.axvline(np.mean(unit_means), color="#e53935", linewidth=1.5,
            label=f"mean = {np.mean(unit_means):.2f}")
    ax.axvline(np.median(unit_means), color="#fb8c00", linewidth=1.5, linestyle="--",
            label=f"median = {np.median(unit_means):.2f}")
    ax.set_xlabel("Average score per overlapping moment (effective=1, partial=0, ineffective=−1)", fontsize=10)
    ax.set_ylabel("Number of key moments (IOU >= 0.3)", fontsize=10)
    ax.set_title(f"{ann_type.capitalize()}: distribution of per-moment average scores  (n={len(unit_means)})", fontsize=11)
    ax.legend(fontsize=9)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.show()

# Overview of annotator trends

In [ ]:
def make_rates_table(ann_type):
    """Return list of (short_id, full_id, rates_dict, total) sorted by ineffective rate desc."""
    rows = []
    for ann_id, type_counts in counts.items():
        if ann_type not in type_counts:
            continue
        c = type_counts[ann_type]
        total = sum(c.values())
        rates = {label: c.get(label, 0) / total for label in LABEL_ORDER}
        rows.append((ann_id[:8], ann_id, rates, total))
    rows.sort(key=lambda r: r[2]["ineffective"], reverse=True)
    return rows


def plot_rates(ann_type, ax):
    rows = make_rates_table(ann_type)
    if not rows:
        ax.set_visible(False)
        return

    short_ids = [r[0] for r in rows]
    totals    = [r[3] for r in rows]
    y = np.arange(len(rows))

    lefts = np.zeros(len(rows))
    for label in LABEL_ORDER:
        vals = np.array([r[2][label] for r in rows])
        bars = ax.barh(y, vals, left=lefts, color=COLORS[label], label=label, height=0.6)
        # Annotate segments ≥ 8% wide
        for bar, v in zip(bars, vals):
            if v >= 0.08:
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_y() + bar.get_height() / 2,
                    f"{v:.0%}",
                    ha="center", va="center", fontsize=10, color="white", fontweight="bold"
                )
        lefts += vals

    # n= labels at the right edge
    for i, (_, _, _, n) in enumerate(rows):
        ax.text(1.01, i, f"n={n}", va="center", fontsize=10, color="#555")

    ax.set_yticks(y)
    ax.set_yticklabels(short_ids, fontsize=10, fontfamily="monospace")
    ax.set_xlim(0, 1)
    ax.set_xlabel("Proportion of annotations", fontsize=9)
    ax.set_title(ann_type.capitalize(), fontsize=11, fontweight="bold")
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
    ax.axvline(0.5, color="#bbb", linewidth=0.8, linestyle="--")
    ax.spines[["top", "right"]].set_visible(False)


fig, axes = plt.subplots(1, 2, figsize=(14, 9))

for ax, ann_type in zip(axes, ["scaffolding", "rapport"]):
    plot_rates(ann_type, ax)

legend_patches = [mpatches.Patch(color=COLORS[l], label=l) for l in LABEL_ORDER]
fig.legend(handles=legend_patches, loc="lower center", ncol=4, fontsize=9,
           frameon=False, bbox_to_anchor=(0.5, -0.01))

fig.suptitle("Per-annotator label rates by annotation type\n(sorted by ineffective rate, annotator IDs truncated to 8 chars)",
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## Threshold-based profile creation

Threshold-based annotator grouping by ineffective rate. 

Annotators are assigned to groups based on their % of ineffective ratings.

Only annotators with > MIN_MOMENTS valid key moments are included.

Then, we calculate IAA (Krippendorff's alpha) for each profile. 

In [ ]:
# --- Configure these ---
# these cutoffs are manually chosen based on distributions in previous plot in an attempt to get good IAA
SCAFFOLDING_CUTOFFS = [0.6, 0.4]   # e.g. [0.3, 0.6] → groups: <30%, 30–60%, ≥60%
RAPPORT_CUTOFFS = [0.2, 0.4]   # e.g. [0.3, 0.6] → groups: <30%, 30–60%, ≥60%
# -----------------------

def _threshold_annotator_vecs(ann_type):
    vecs = {}
    for ann_id, tc in counts.items():
        if ann_type not in tc:
            continue
        c     = tc[ann_type]
        total = sum(v for k, v in c.items() if k in VALID_LABELS)
        if total <= MIN_MOMENTS:
            continue
        vecs[ann_id] = {
            "ineffective": c.get("ineffective", 0) / total,
            "total":       total,
        }
    return vecs

for ann_type in ["scaffolding", "rapport"]:
    if ann_type == "scaffolding":
        cutoffs = SCAFFOLDING_CUTOFFS
    else:
        cutoffs = RAPPORT_CUTOFFS
    cutoffs = [0.0] + sorted(cutoffs) + [1.01]
    vecs              = _threshold_annotator_vecs(ann_type)
    full_mat, all_ann = get_reliability_matrix(ann_type)

    cutoff_labels = " | ".join(
        f"[{cutoffs[i]:.0%}, {'100%' if cutoffs[i+1] > 1 else f'{cutoffs[i+1]:.0%}'})"
        for i in range(len(cutoffs) - 1)
    )
    print(f"\n{'='*60}")
    print(f"{ann_type.upper()}  (MIN_MOMENTS > {MIN_MOMENTS}, IoU >= {IOU_THRESHOLD})")
    print(f"Groups by ineffective rate: {cutoff_labels}")

    for i in range(len(cutoffs) - 1):
        lo, hi      = cutoffs[i], cutoffs[i + 1]
        hi_label    = "100%" if hi > 1.0 else f"{hi:.0%}"
        group_label = f"[{lo:.0%}, {hi_label})"
        members     = [a for a, v in vecs.items() if lo <= v["ineffective"] < hi]

        if not members:
            print(f"\n  Group {i+1} {group_label}: no annotators")
            continue

        mat     = submatrix_for(full_mat, all_ann, members)
        alpha   = compute_iaa(mat)
        n_units = mat.shape[1] if mat is not None else 0

        alpha_str      = f"{alpha:.4f}" if alpha is not None else "n/a (no shared units)"
        members_display = ", ".join(sorted(a[:8] for a in members))

        print(f"\n  Group {i+1} {group_label}: {len(members)} annotators, {n_units} overlapping moments")
        print(f"    Annotators:       {members_display}")
        print(f"    Krippendorff's α: {alpha_str}")


# What if we derived profiles by clustering? 

K-medoids clustering of annotator vectors using Earth Mover's Distance (EMD). 

Each annotator is represented as (effective%, partial%, ineffective%), normalised over those three labels only (unclear excluded).

Only annotators with >= MIN_MOMENTS valid key moments are included. EMD treats the label vector as a probability distribution over the ordinal scale effective=0, partial=1, ineffective=2. This respects the ordering between labels, unlike Euclidean or cosine distance.

K-medoids is used because medoids must be actual data points (required when using a custom distance matrix).

**Takeaway**: we don't gain much from this more complicated approach over threshold-based profiles.

In [ ]:
K_VALUES        = [2, 3, 4]
N_INIT          = 20
CLUSTER_PALETTE = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]

SQRT3_2           = np.sqrt(3) / 2
CORNERS           = np.array([[0.5, SQRT3_2], [0.0, 0.0], [1.0, 0.0]])
ORDINAL_POSITIONS = [0, 1, 2]


def emd(p, q):
    return wasserstein_distance(ORDINAL_POSITIONS, ORDINAL_POSITIONS,
                                u_weights=p, v_weights=q)


def pairwise_emd(vecs):
    n = len(vecs)
    D = np.zeros((n, n))
    for i in range(n):
        for j in range(i + 1, n):
            d = emd(vecs[i], vecs[j])
            D[i, j] = D[j, i] = d
    return D


def kmedoids_once(D, k, rng):
    n       = D.shape[0]
    medoids = [rng.integers(n)]
    for _ in range(k - 1):
        dists = np.min(D[:, medoids], axis=1)
        probs = dists / dists.sum()
        medoids.append(rng.choice(n, p=probs))
    medoids = list(dict.fromkeys(medoids))
    for _ in range(100):
        labels      = np.argmin(D[:, medoids], axis=1)
        new_medoids = []
        for ci in range(k):
            members = np.where(labels == ci)[0]
            if len(members) == 0:
                new_medoids.append(medoids[ci])
            else:
                new_medoids.append(members[np.argmin(D[np.ix_(members, members)].sum(axis=1))])
        if new_medoids == medoids:
            break
        medoids = new_medoids
    labels     = np.argmin(D[:, medoids], axis=1)
    total_cost = sum(D[i, medoids[labels[i]]] for i in range(n))
    return labels, total_cost


def kmedoids(D, k, n_init=N_INIT, seed=42):
    rng        = np.random.default_rng(seed)
    best_labels, best_cost = None, np.inf
    for _ in range(n_init):
        labels, cost = kmedoids_once(D, k, rng)
        if cost < best_cost:
            best_cost, best_labels = cost, labels
    return best_labels


def build_annotator_vectors(ann_type):
    vecs = {}
    for ann_id, type_counts in counts.items():
        if ann_type not in type_counts:
            continue
        c     = type_counts[ann_type]
        total = sum(v for k, v in c.items() if k in VALID_LABELS)
        if total < MIN_MOMENTS:
            continue
        vecs[ann_id] = np.array([
            c.get("effective",   0) / total,
            c.get("partial",     0) / total,
            c.get("ineffective", 0) / total,
        ])
    return vecs


def to_cartesian(eff, partial, ineff):
    return 0.5 * eff + ineff, eff * SQRT3_2


def draw_ternary_frame(ax):
    ax.add_patch(plt.Polygon(CORNERS, fill=False, edgecolor="#333", linewidth=1.2))
    for t in [0.25, 0.50, 0.75]:
        for p1, p2 in [
            (to_cartesian(t, 1-t, 0), to_cartesian(t, 0, 1-t)),
            (to_cartesian(1-t, t, 0), to_cartesian(0, t, 1-t)),
            (to_cartesian(1-t, 0, t), to_cartesian(0, 1-t, t)),
        ]:
            ax.plot([p1[0], p2[0]], [p1[1], p2[1]], color="#ddd", linewidth=0.5, zorder=0)
    off = 0.08
    ax.text(CORNERS[0][0], CORNERS[0][1] + off, "Eff",   ha="center", fontsize=9,
            fontweight="bold", color=COLORS["effective"])
    ax.text(CORNERS[1][0] - off, CORNERS[1][1] - 0.03, "Par",   ha="center", fontsize=9,
            fontweight="bold", color=COLORS["partial"])
    ax.text(CORNERS[2][0] + off, CORNERS[2][1] - 0.03, "Ineff", ha="center", fontsize=9,
            fontweight="bold", color=COLORS["ineffective"])


def plot_clustered_ternary(ann_type, k, vecs, assignment, full_mat, all_ann, ax):
    draw_ternary_frame(ax)

    ann_ids  = list(vecs.keys())
    clusters = [assignment[a] for a in ann_ids]
    xs, ys, texts = [], [], []

    for ann_id, cluster_id in zip(ann_ids, clusters):
        eff, par, ineff = vecs[ann_id]
        x, y = to_cartesian(eff, par, ineff)
        xs.append(x); ys.append(y)
        n_total = sum(v for lbl, v in counts[ann_id][ann_type].items() if lbl in VALID_LABELS)
        ax.scatter(x, y, s=n_total * 0.4 + 40, color=CLUSTER_PALETTE[cluster_id],
                   alpha=0.9, zorder=3, edgecolors="white", linewidths=0.5)
        t = ax.text(x, y, ann_id[:8], ha="center", va="bottom",
                    fontsize=10, fontfamily="monospace", color="#222", zorder=4)
        texts.append(t)

    adjust_text(texts, x=xs, y=ys, ax=ax, expand=(1.4, 1.6),
                arrowprops=dict(arrowstyle="-", color="#aaa", lw=0.5))

    kappa_lines = []
    for cid in range(k):
        members  = [a for a, cl in zip(ann_ids, clusters) if cl == cid]
        mat      = submatrix_for(full_mat, all_ann, members)
        alpha    = compute_iaa(mat)
        n_units  = mat.shape[1] if mat is not None else 0
        alpha_str = f"{alpha:.2f}" if alpha is not None else "n/a"
        kappa_lines.append(f"C{cid+1} (n={len(members)}): α={alpha_str} [{n_units} moments]")

    ax.text(0.5, -0.07, "\n".join(kappa_lines),
            ha="center", va="top", fontsize=10, fontfamily="monospace",
            color="#333", linespacing=1.6,
            bbox=dict(boxstyle="round,pad=0.3", facecolor="#f5f5f5",
                      edgecolor="#ccc", linewidth=0.8))

    ax.set_xlim(-0.18, 1.18)
    ax.set_ylim(-0.28, SQRT3_2 + 0.18)
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title(f"k={k}", fontsize=15, fontweight="bold", pad=10)


fig, axes = plt.subplots(2, len(K_VALUES), figsize=(6 * len(K_VALUES), 13))

for row, ann_type in enumerate(["scaffolding", "rapport"]):
    vecs          = build_annotator_vectors(ann_type)
    full_mat, all_ann = get_reliability_matrix(ann_type)
    ann_ids       = list(vecs.keys())
    X             = np.array([vecs[a] for a in ann_ids])
    D             = pairwise_emd(X)

    print(f"{ann_type}: {len(ann_ids)} annotators with >= {MIN_MOMENTS} moments")

    for col, k in enumerate(K_VALUES):
        ax          = axes[row][col]
        effective_k = min(k, len(ann_ids))
        labels      = kmedoids(D, effective_k)
        assignment  = {a: int(labels[i]) for i, a in enumerate(ann_ids)}
        plot_clustered_ternary(ann_type, effective_k, vecs, assignment, full_mat, all_ann, ax)
        if col == 0:
            ax.set_ylabel(ann_type.capitalize(), fontsize=15, fontweight="bold", labelpad=70)

fig.suptitle(
    f"K-medoids (Earth Mover's Distance) + intra-cluster Krippendorff's α\n"
    f"(annotators with ≥ {MIN_MOMENTS} moments; IoU ≥ {IOU_THRESHOLD} for unit clustering)",
    fontsize=15, y=1.01,
)
legend_handles = [mpatches.Patch(color=CLUSTER_PALETTE[i], label=f"Cluster {i+1}")
                  for i in range(max(K_VALUES))]
fig.legend(handles=legend_handles, loc="lower center", ncol=max(K_VALUES),
           fontsize=15, frameon=False, bbox_to_anchor=(0.5, -0.01))
plt.tight_layout()
plt.show()


# Individual annotators' IAA

Pairwise IAA ranking: Krippendorff's α (ordinal) for all annotator pairs, if they have at least MIN_OVERLAP IoU-clustered moments.

I also tried plotting a network graph based on IAA but it was a mess so I removed it.

In [ ]:
MIN_OVERLAP = 50

for ann_type in ["scaffolding", "rapport"]:
    full_mat, all_ann = get_reliability_matrix(ann_type)
    if full_mat.size == 0:
        print(f"{ann_type}: no data")
        continue

    rows = []
    for i, a1 in enumerate(all_ann):
        for a2 in all_ann[i + 1:]:
            mat = submatrix_for(full_mat, all_ann, [a1, a2])
            if mat is None:
                continue
            n_units = mat.shape[1]
            if n_units < MIN_OVERLAP:
                continue
            alpha = compute_iaa(mat)
            if alpha is None or np.isnan(alpha):
                continue
            rows.append({
                "annotator A": a1[:8],
                "annotator B": a2[:8],
                "α":           alpha,
                "n units":     n_units,
            })

    df = (pd.DataFrame(rows)
            .sort_values("α", ascending=False)
            .reset_index(drop=True))
    df.index += 1
    df.index.name = "rank"

    print(f"\n{ann_type.upper()} — Krippendorff's α (n >= {MIN_OVERLAP} shared moments, {len(df)} pairs)")
    print(df.to_string())


# Looking at actual annotations

Disagreement examples: IoU-clustered units where annotators gave different labels.

Shows all annotations for the same key moment.

In [ ]:
TRANSCRIPT_PATH = REPO_ROOT / "data" / "transcripts" / "step_up.jsonl"
N_EXAMPLES      = 10    # per annotation type
MAX_TURN_CHARS  = 180

LABEL_COLORS = {
    "effective":   "#2e7d32",
    "partial":     "#e65100",
    "ineffective": "#b71c1c",
}


def find_disagreement_units(ann_type):
    """Return units (IoU-clustered moment groups) where annotators disagree."""
    units = []
    for f in GT_DIR.glob("*.json"):
        with open(f) as fp:
            d = json.load(fp)
        conv_id = d["conversation_id"]
        moments = [
            m for m in d.get("key_moments", [])
            if m.get("annotation_type") == ann_type
            and m.get("strategy_label") in VALID_LABELS
        ]
        for cluster in _cluster_moments(moments):
            seen = {}
            for m in cluster:
                if m["annotator_id"] not in seen:
                    seen[m["annotator_id"]] = m
            annotations = list(seen.values())
            if len(annotations) < 2:
                continue
            if len({m["strategy_label"] for m in annotations}) == 1:
                continue  # all agree
            units.append({"conv_id": conv_id, "annotations": annotations})
    return units


def sample_units(units, n):
    """Sample to cover different label-set combinations, then fill by unit size."""
    seen_combos = set()
    selected    = []
    for unit in sorted(units, key=lambda u: -len(u["annotations"])):
        combo = frozenset(m["strategy_label"] for m in unit["annotations"])
        if combo not in seen_combos:
            seen_combos.add(combo)
            selected.append(unit)
        if len(selected) >= n:
            break
    for unit in units:
        if len(selected) >= n:
            break
        if unit not in selected:
            selected.append(unit)
    return selected


def load_turns(conv_ids):
    needed = set(conv_ids)
    result = {}
    with open(TRANSCRIPT_PATH) as f:
        for line in f:
            if not line.strip():
                continue
            r = json.loads(line)
            if r["transcript_id"] in needed:
                result[r["transcript_id"]] = r["turns"]
                if len(result) == len(needed):
                    break
    return result


def render_unit(unit, turns_by_conv, ann_type):
    conv_id     = unit["conv_id"]
    annotations = unit["annotations"]

    t_start = min(m["turn_start"] for m in annotations)
    t_end   = max(m["turn_end"]   for m in annotations)

    turns        = turns_by_conv.get(conv_id, [])
    excerpt_rows = ""
    for t in turns:
        if t_start <= t["turn_number"] <= t_end:
            role  = t["role"]
            text  = t["text"][:MAX_TURN_CHARS] + ("\u2026" if len(t["text"]) > MAX_TURN_CHARS else "")
            color = "#1565c0" if role == "Tutor" else "#4a148c"
            excerpt_rows += (
                f'<tr><td style="color:{color};font-weight:bold;white-space:nowrap;'
                f'padding:2px 8px 2px 0;vertical-align:top">{role}</td>'
                f'<td style="padding:2px 0;color:#333">{text}</td></tr>'
            )

    def ann_card(m, idx):
        lbl       = m["strategy_label"]
        color     = LABEL_COLORS.get(lbl, "#555")
        badge     = (f'<span style="background:{color};color:white;padding:2px 8px;'
                     f'border-radius:3px;font-size:12px">{lbl}</span>')
        turns_str = f"turns {m['turn_start']}\u2013{m['turn_end']}"
        return (
            f'<div style="flex:1;min-width:200px;background:#f9f9f9;border:1px solid #ddd;'
            f'border-radius:6px;padding:10px;margin:4px">'
            f'<div style="font-size:11px;color:#888;margin-bottom:4px">'
            f'Annotator {idx} \u00b7 {turns_str}</div>'
            f'{badge}'
            f'<p style="margin:8px 0 4px 0"><b>Situation:</b> {m.get("situation","") or "<em>\u2014</em>"}</p>'
            f'<p style="margin:4px 0"><b>Action:</b> {m.get("action","") or "<em>\u2014</em>"}</p>'
            f'<p style="margin:4px 0"><b>Result:</b> {m.get("result","") or "<em>\u2014</em>"}</p>'
            f'</div>'
        )

    label_summary = ", ".join(
        '<span style="color:{};font-weight:bold">{}</span>'.format(
            LABEL_COLORS.get(m["strategy_label"], "#555"), m["strategy_label"]
        )
        for m in annotations
    )
    cards = "".join(ann_card(m, i + 1) for i, m in enumerate(annotations))

    return (
        f'<div style="border:1px solid #bbb;border-radius:8px;padding:14px;margin:16px 0;'
        f'font-family:sans-serif;font-size:13px">'
        f'<div style="font-size:11px;color:#777;margin-bottom:8px">'
        f'<b>{ann_type}</b> \u00b7 conv <code>{conv_id[:8]}</code> \u00b7 '
        f'{len(annotations)} annotators \u00b7 labels: {label_summary}</div>'
        f'<details><summary style="cursor:pointer;font-weight:bold;margin-bottom:6px">'
        f'Transcript turns {t_start}\u2013{t_end}</summary>'
        f'<table style="width:100%;border-collapse:collapse;margin:6px 0 10px 0;font-size:12px">'
        f'{excerpt_rows}</table></details>'
        f'<div style="display:flex;flex-wrap:wrap;gap:4px">{cards}</div>'
        f'</div>'
    )


html_parts = [
    "<h2 style='font-family:sans-serif'>Disagreement examples — all annotators per unit</h2>"
]

for ann_type in ["scaffolding", "rapport"]:
    units    = find_disagreement_units(ann_type)
    examples = sample_units(units, N_EXAMPLES)
    turns_by_conv = load_turns([e["conv_id"] for e in examples])

    html_parts.append(
        f"<h3 style='font-family:sans-serif;border-bottom:2px solid #ccc;padding-bottom:4px'>"
        f"{ann_type.capitalize()} ({len(units):,} disagreement units)</h3>"
    )
    for ex in examples:
        html_parts.append(render_unit(ex, turns_by_conv, ann_type))

display(HTML("\n".join(html_parts)))
